# Day12：Safe Local Git Agent

## Goal

在完全临时的 Git 仓库中验证任务分支、批准文件提交、脏基线拒绝和批准后哈希漂移拒绝。Notebook 不调用 LLM，也不接触真实项目历史。

## Setup

加载 Day12 组件，并定义可复用的临时仓库辅助函数。

In [1]:
import hashlib
import os
from pathlib import Path
import subprocess
import sys
import tempfile

project_root = Path.cwd().parent if Path.cwd().name == 'day12' else Path.cwd()
sys.path.insert(0, str(project_root))

from agents.git import GitAgent
from tools.git_tool import GitTool

def run_git(repository, *arguments):
    return subprocess.run(
        ['git', *arguments], cwd=repository, check=True,
        text=True, encoding='utf-8', errors='replace', capture_output=True,
    ).stdout.strip()

def write_text(repository, name, content):
    path = Path(repository, name)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8', newline='')

def create_repository():
    temporary = tempfile.TemporaryDirectory()
    repository = temporary.name
    run_git(repository, 'init', '-b', 'main')
    run_git(repository, 'config', 'user.name', 'Day12 Notebook')
    run_git(repository, 'config', 'user.email', 'day12@example.com')
    write_text(repository, 'Inventory.cs', 'class Inventory {}\n')
    run_git(repository, 'add', '--', 'Inventory.cs')
    run_git(repository, 'commit', '-m', 'chore: baseline')
    return temporary, repository

def content_hash(content):
    return hashlib.sha256(content.encode('utf-8')).hexdigest()

print('Day12 setup ready')

Day12 setup ready


## Steps

### 1. 创建任务分支并提交批准文件

In [2]:
success_temp, success_repo = create_repository()
success_agent = GitAgent(GitTool(success_repo), id_factory=lambda: 'notebook1')
prepared = success_agent.prepare({})
assert prepared['git_status'] == 'prepared'
assert prepared['git_branch'] == 'agent/notebook1'

approved_content = 'class Inventory { int Capacity; }\n'
write_text(success_repo, 'Inventory.cs', approved_content)
commit_state = {
    **prepared,
    'approved_changes': [{
        'file': 'Inventory.cs',
        'operation': 'modify',
        'after_hash': content_hash(approved_content),
    }],
    'code_check_result': {'success': True},
    'compile_result': {'success': True},
    'test_result': {'success': True},
    'review': {'pass': True, 'score': 95, 'remaining_issues': []},
    'approval_history': [{'source': 'coder', 'status': 'approved'}],
}
committed = success_agent.commit(commit_state)
assert committed['git_status'] == 'committed'
assert committed['git_result']['files'] == ['Inventory.cs']
print({
    'branch': prepared['git_branch'],
    'commit': committed['git_result']['commit_hash'],
    'message': committed['git_result']['message'],
})

{'branch': 'agent/notebook1', 'commit': 'b64b648f2da71239eb9faab18c295cc0a661d7b1', 'message': 'feat: 提交已批准的 AI 代码变更'}


### 2. 拒绝包含用户改动的脏基线

In [3]:
dirty_temp, dirty_repo = create_repository()
write_text(dirty_repo, 'user-note.txt', 'local work\n')
dirty_result = GitAgent(GitTool(dirty_repo), id_factory=lambda: 'dirty1').prepare({})
assert dirty_result['git_status'] == 'error'
assert dirty_result['git_result']['error_code'] == 'DIRTY_BASELINE'
print(dirty_result['git_result']['error_code'])

DIRTY_BASELINE


### 3. 拒绝批准后的文件漂移

In [4]:
drift_temp, drift_repo = create_repository()
drift_agent = GitAgent(GitTool(drift_repo), id_factory=lambda: 'drift1')
drift_prepared = drift_agent.prepare({})
write_text(drift_repo, 'Inventory.cs', 'external edit\n')
drift_state = {
    **commit_state,
    **drift_prepared,
    'approved_changes': [{
        'file': 'Inventory.cs', 'operation': 'modify',
        'after_hash': content_hash('approved content\n'),
    }],
}
drift_result = drift_agent.commit(drift_state)
assert drift_result['git_status'] == 'error'
assert drift_result['git_result']['error_code'] == 'APPROVED_CONTENT_DRIFT'
print(drift_result['git_result']['error_code'])

APPROVED_CONTENT_DRIFT


## Checks

清理临时仓库，并确认三条关键安全路径都已执行。

In [5]:
for temporary in (success_temp, dirty_temp, drift_temp):
    temporary.cleanup()

assert committed['git_result']['success']
assert dirty_result['git_result']['error_code'] == 'DIRTY_BASELINE'
assert drift_result['git_result']['error_code'] == 'APPROVED_CONTENT_DRIFT'
print('Day12 no-LLM acceptance passed')

Day12 no-LLM acceptance passed


## Next Steps

Day12 只创建本地任务分支与提交。远程 push、Pull Request、合并、变基、暂存用户改动和历史改写均不在范围内。